# Try Alternative transformer models
-  Distilbert for its efficiency
-  Legalbert for domain-specific

## Import Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets scikit-learn tqdm joblib nltk --quiet

import ast
import logging
import os
import pickle
import random
import re
import shutil
import sys
import time
import warnings
from collections import Counter
from itertools import chain
from pathlib import Path
import chardet
import joblib
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from tqdm import tqdm
from tqdm.auto import tqdm
from pathlib import Path
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split

from transformers import (AutoTokenizer,AutoModelForSequenceClassification, BertForSequenceClassification,
                          BertTokenizerFast, DistilBertForSequenceClassification,
                          DistilBertTokenizerFast, logging as transformers_logging,
                          get_linear_schedule_with_warmup)

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

from openpyxl import load_workbook
import glob






Mounted at /content/drive


## Load dataset and preprocess

In [2]:
# Paths to data folders
annotations_dir = '/content/drive/MyDrive/Colab Notebooks/Project/English_consolidation/'
texts_dir = '/content/drive/MyDrive/Colab Notebooks/Project/English_sanitized_policies/'


# List .csv or .tsv files in the annotations_dir
csv_files = [f for f in os.listdir(annotations_dir) if f.endswith(('.csv', '.tsv'))]

sample_csv = pd.read_csv(os.path.join(annotations_dir, csv_files[0]), sep='\t', engine='python')
print(sample_csv.columns.tolist())


['policy id,segment_id,category name,attribute name,value name,policy_type,privacy_policy_link,MAPP_59']


In [3]:
# Create an empty list to store all DataFrames
all_dfs = []

# Loop through each CSV to match its TXT
for csv_file in tqdm(csv_files, desc="Processing files"):
    csv_path = os.path.join(annotations_dir, csv_file)
    txt_file = csv_file.replace('.csv', '.txt')
    txt_path = os.path.join(texts_dir, txt_file)

    # Checking if the matching text file exists
    if not os.path.exists(txt_path):
        print(f"Text file missing for: {csv_file}")
        continue

    # Load CSV
    df = pd.read_csv(csv_path, sep=',', engine='python')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # Read and segment the text file (spliting by double newlines)
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()
        segments = [seg.strip() for seg in text.replace('\r\n', '\n').replace('\r', '\n').split('\n\n') if seg.strip()]

    # Map segment text to DataFrame
    df['segment_text'] = df['segment_id'].apply(
        lambda x: segments[x - 1] if 1 <= x <= len(segments) else ""
    )

    df['source_file'] = csv_file
    all_dfs.append(df)

# Combine all into df
final_df = pd.concat(all_dfs, ignore_index=True)
print("\n All files processed. Final DataFrame shape:", final_df.shape)


Processing files:   0%|          | 0/64 [00:00<?, ?it/s]


 All files processed. Final DataFrame shape: (3976, 10)


In [4]:
# Extract APP_ID from source_file in final_df
final_df['APP_ID'] = final_df['source_file'].str.extract(r'_(.*)\.csv')

final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3976 entries, 0 to 3975
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   policy_id            3976 non-null   int64 
 1   segment_id           3976 non-null   int64 
 2   category_name        3976 non-null   object
 3   attribute_name       3976 non-null   object
 4   value_name           3976 non-null   object
 5   policy_type          3976 non-null   object
 6   privacy_policy_link  3976 non-null   object
 7   mapp_59              3976 non-null   bool  
 8   segment_text         3976 non-null   object
 9   source_file          3976 non-null   object
 10  APP_ID               3976 non-null   object
dtypes: bool(1), int64(2), object(8)
memory usage: 314.6+ KB


In [5]:
# Save merged data
final_df.to_csv("mapp_preprocessed_dataset2.csv", index=False)

# Check data
print("Unique policies:", final_df['policy_id'].nunique())

missing_segments = final_df[final_df['segment_text'].isnull()]
print("Total missing segments:", len(missing_segments))

duplicates =  final_df[final_df.duplicated(subset=['policy_id', 'segment_id', 'category_name', 'attribute_name', 'segment_text'], keep=False)]
print("Total duplicates:", len(duplicates))

# Check total segments
text_files = [f for f in os.listdir(texts_dir) if f.endswith('.txt')]

total_segments = 0

for txt_file in sorted(text_files):
    with open(os.path.join(texts_dir, txt_file), 'r', encoding='utf-8') as f:
        text = f.read()
        segments = [seg.strip() for seg in re.split(r'\n\s*\n', text) if seg.strip()]
        num_segments = len(segments)
        total_segments += num_segments

print(f"Total segments across all TXT files: {total_segments}")


Unique policies: 64
Total missing segments: 0
Total duplicates: 0
Total segments across all TXT files: 3976


In [6]:
sam = final_df.copy()

# list of all unique value strings
sam['value_name'] = sam['value_name'].apply(lambda x: ast.literal_eval(x))
all_labels = set()
for labels in sam['value_name']:
    all_labels.update(labels)
all_labels = sorted(list(all_labels))
all_labels

['Anonymization (opt)_Aggregated or anonymized',
 'Anonymization (opt)_Identifiable',
 'Choice Scope (opt)_Use',
 'Choice Type (opt)_Browser/device privacy controls',
 "Choice Type (opt)_Don't use service/feature",
 'Choice Type (opt)_First-party privacy controls',
 'Choice Type (opt)_Opt-in',
 'Choice Type (opt)_Opt-out link',
 'Choice Type (opt)_Opt-out via contacting company',
 'Choice Type (opt)_Other',
 'Choice Type (opt)_Third-party privacy controls',
 'Collection Mode (opt)_Explicit',
 'Collection Mode (opt)_Implicit',
 'Collection Process_Collected on first-party website/app',
 'Collection Process_Directly received from third party',
 'Collection Process_Entered by user on first-party website/app',
 'Collection Process_Entered by user on third-party website/app',
 'Collection Process_Other',
 'Collection Process_Shared by first party with a third party',
 'Collection Process_Tracked on first-party website/app by third party',
 'Collection Process_Tracked user on third-party web

In [7]:
sam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3976 entries, 0 to 3975
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   policy_id            3976 non-null   int64 
 1   segment_id           3976 non-null   int64 
 2   category_name        3976 non-null   object
 3   attribute_name       3976 non-null   object
 4   value_name           3976 non-null   object
 5   policy_type          3976 non-null   object
 6   privacy_policy_link  3976 non-null   object
 7   mapp_59              3976 non-null   bool  
 8   segment_text         3976 non-null   object
 9   source_file          3976 non-null   object
 10  APP_ID               3976 non-null   object
dtypes: bool(1), int64(2), object(8)
memory usage: 314.6+ KB


## Train Distilbert

In [ ]:
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATES = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# directory
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
excel_path = "/content/drive/MyDrive/Colab Notebooks/Project/training_results_distilbert.xlsx"

# for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Filter and prepare
sam = sam[sam['value_name'].apply(lambda x: len(x) > 0)]
unique_labels = sorted(set(label for sublist in sam['value_name'] for label in sublist))

custom_labels = [
    'Purpose_Essential service or feature',
    "Collection Process_Shared by first party with a third party",
    "Information Type_IP address and device IDs",

    #'Information Type_Contact information',
    #"Information Type_Location",
    #"Information Type_Health, genetic, or biometric data",

    #"Information Type_Computer information",
    #"Information Type_User online activities",
    #"Information Type_Generic personal information",

    #"Collection Process_Collected on first-party website/app",
    #"Purpose_Advertising or marketing",
    #'Information Type_Personal identifier',

    #"Purpose_Analytics or research",
    #"Purpose_Service operation and security",
    #"Purpose_Legal requirement"

]


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits.squeeze())
            preds.extend(probs.cpu().numpy())

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Optimize with higher lr
def get_optimizer(model, base_lr, classifier_lr=1e-4, weight_decay=0.01):
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': 0.0
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': 0.0
        },
    ]
    return AdamW(optimizer_grouped_parameters)

# Train
def train_model(lr, train_loader, pos_weight_val):
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([pos_weight_val]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"LR {lr:.0e} | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Final retrain
def retrain_best_model(best_lr, best_pos_weight, full_train_loader):
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=best_lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([best_pos_weight]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(full_train_loader, desc=f"Retrain | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Training Loop
all_results = []

for label_to_train in custom_labels:
    safe_label = label_to_train.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_distilbert_model_{safe_label}.pt"

    if model_path.exists():
        print(f" Skipping {label_to_train} — model already exists.")
        continue

    print(f"\nTraining for: {label_to_train}")

    train_df = sam[sam['policy_type'] == 'TRAIN'].copy()
    train_df['binary_label'] = train_df['value_name'].apply(lambda x: 1 if label_to_train in x else 0)

    positives = train_df[train_df['binary_label'] == 1]
    negatives = train_df[train_df['binary_label'] == 0]
    if len(positives) < 5:
        print(f" Too few positives ({len(positives)}). Skipping.")
        continue

    # pos_weight on original training data
    label_counts = Counter(train_df['binary_label'].tolist())
    pos_weight = label_counts[0] / label_counts[1]
    print(f" Using pos_weight: {pos_weight:.2f}")

    # Train/val split
    texts = train_df['segment_text'].tolist()
    labels = train_df['binary_label'].tolist()
    texts_train, texts_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"

    with open(val_save_path, "wb") as f:
        pickle.dump({"texts": texts_val, "labels": y_val}, f)

    # Oversample positives
    train_data = pd.DataFrame({'segment_text': texts_train, 'binary_label': y_train})
    positives_train = train_data[train_data['binary_label'] == 1]
    negatives_train = train_data[train_data['binary_label'] == 0]

    max_oversample_size = min(len(negatives_train), len(positives_train) * 4)
    oversampled_positives = positives_train.sample(max_oversample_size, replace=True, random_state=SEED)
    train_df_balanced = pd.concat([negatives_train, oversampled_positives], ignore_index=True).sample(frac=1, random_state=SEED)

    # loaders
    train_dataset = TextDataset(train_df_balanced['segment_text'].tolist(), train_df_balanced['binary_label'].tolist())
    val_dataset = TextDataset(texts_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Find best model
    best_f1 = 0
    best_lr = None
    for lr in LEARNING_RATES:
        model = train_model(lr, train_loader, pos_weight)
        acc, f1, precision, recall = evaluate_model(model, val_loader)

        if f1 > best_f1:
            best_f1 = f1
            best_lr = lr
            torch.save(model.state_dict(), model_path)
            print(f"Best model updated and saved at LR {lr} for label {label_to_train}")

        all_results.append({
            'label': label_to_train,
            'learning_rate': lr,
            'pos_weight': round(pos_weight, 2),
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'best_model': 'best' if f1 == best_f1 else ''
        })

        # Save results to excel after every LR
        all_models_df = pd.DataFrame(all_results)

        # Append or create "All Models" sheet
        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "All Models" in book.sheetnames:
                existing_df = pd.read_excel(excel_path, sheet_name="All Models")
                combined_df = pd.concat([existing_df, all_models_df], ignore_index=True)
                combined_df.drop_duplicates(subset=['label', 'learning_rate'], inplace=True)
            else:
                combined_df = all_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_df.to_excel(writer, sheet_name="All Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                all_models_df.to_excel(writer, sheet_name="All Models", index=False)

        # Append or create "Best Models" sheet
        best_models_df = all_models_df.sort_values('f1', ascending=False).drop_duplicates('label')

        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "Best Models" in book.sheetnames:
                existing_best_df = pd.read_excel(excel_path, sheet_name="Best Models")
                combined_best_df = pd.concat([existing_best_df, best_models_df], ignore_index=True)
                combined_best_df.drop_duplicates(subset=['label'], inplace=True)
            else:
                combined_best_df = best_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_best_df.to_excel(writer, sheet_name="Best Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                best_models_df.to_excel(writer, sheet_name="Best Models", index=False)



    # Final retrain on full dataset
    full_dataset = TextDataset(train_df['segment_text'].tolist(), train_df['binary_label'].tolist())
    full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n Retraining best model for {label_to_train} on full data...")
    best_model = retrain_best_model(best_lr, pos_weight, full_loader)

    torch.save(best_model.state_dict(), model_path)
    print(f" Final model saved to {model_path}")


 Skipping Purpose_Essential service or feature — model already exists.
 Skipping Collection Process_Shared by first party with a third party — model already exists.
 Skipping Information Type_IP address and device IDs — model already exists.


## Test Distilbert





In [ ]:
# Config
MAX_LEN = 512
BATCH_SIZE = 32
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/models")
TEST_EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Project/models/test_results_distilbert.xlsx"

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')


custom_labels = [
    'Purpose_Essential service or feature',
    "Collection Process_Shared by first party with a third party",
    "Information Type_IP address and device IDs"

]

# convert text and labels into token IDs, attention masks, and tensors for training
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }
#Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).squeeze(-1)
            probs = probs.detach().cpu().numpy().flatten()
            preds.extend(probs)

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

#Load models, test and save results
results = []

for label_to_test in tqdm(custom_labels, desc="Testing all labels"):
    safe_label = label_to_test.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_distilbert_model_{safe_label}.pt"

    if not model_path.exists():
        print(f" Model file missing for {label_to_test}, skipping.")
        continue

    print(f"\n Evaluating model for label: {label_to_test}")

    # Prepare test data
    test_df = sam[sam['policy_type'] == 'TEST'].copy()
    test_df['binary_label'] = test_df['value_name'].apply(lambda x: 1 if label_to_test in x else 0)

    test_dataset = TextDataset(test_df['segment_text'].tolist(), test_df['binary_label'].tolist())
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # Load model
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    acc, f1, precision, recall = evaluate_model(model, test_loader)
    print(f" Results: Acc={acc:.4f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}\n")

    results.append({
        'label': label_to_test,
        'accuracy': round(acc, 4),
        'f1_score': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
    })

# Save results
dist_results_df = pd.DataFrame(results)
dist_results_df.to_excel(TEST_EXCEL_PATH, index=False)
print(f"\n Test results saved to: {TEST_EXCEL_PATH}")




dist_results_df

Testing all labels:   0%|          | 0/3 [00:00<?, ?it/s]


 Evaluating model for label: Purpose_Essential service or feature


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

 Results: Acc=0.7897, F1=0.5612, Precision=0.4643, Recall=0.7091


 Evaluating model for label: Collection Process_Shared by first party with a third party


...

 Results: Acc=0.8414, F1=0.5577, Precision=0.5179, Recall=0.6042


 Evaluating model for label: Information Type_IP address and device IDs


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

 Results: Acc=0.9517, F1=0.8108, Precision=0.8824, Recall=0.7500


 Test results saved to: /content/drive/MyDrive/Colab Notebooks/Project/models/test_results_distilbert.xlsx


,label,accuracy,f1_score,precision,recall
0,Purpose_Essential service or feature,0.7897,0.5612,0.4643,0.7091
1,Collection Process_Shared by first party with ...,0.8414,0.5577,0.5179,0.6042
2,Information Type_IP address and device IDs,0.9517,0.8108,0.8824,0.7500


In [ ]:
test_df_full = pd.DataFrame()
for label in custom_labels:
    test_df = sam[sam['policy_type'] == 'TEST'].copy()
    test_df['binary_label'] = test_df['value_name'].apply(lambda x: 1 if label in x else 0)
    print(label, test_df['binary_label'].value_counts())


print(test_df_full.shape)

Purpose_Essential service or feature binary_label
0    235
1     55
Name: count, dtype: int64
Collection Process_Shared by first party with a third party binary_label
0    242
1     48
Name: count, dtype: int64
Information Type_IP address and device IDs binary_label
0    250
1     40
Name: count, dtype: int64
(0, 0)


## Train Legalbert

In [ ]:
# Config
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATES = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
excel_path = "/content/drive/MyDrive/Colab Notebooks/Project/training_results_legalbert.xlsx"

# for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained('nlpaueb/legal-bert-base-uncased')

# Filter and prepare
sam = sam[sam['value_name'].apply(lambda x: len(x) > 0)]
unique_labels = sorted(set(label for sublist in sam['value_name'] for label in sublist))

custom_labels = [
    'Purpose_Essential service or feature',
    "Collection Process_Shared by first party with a third party",
    "Information Type_IP address and device IDs",

    #'Information Type_Contact information',
    #"Information Type_Location",
    #"Information Type_Health, genetic, or biometric data",

    #"Information Type_Computer information",
    #"Information Type_User online activities",
    #"Information Type_Generic personal information",

    #"Collection Process_Collected on first-party website/app",
    #"Purpose_Advertising or marketing",
    #'Information Type_Personal identifier',

    #"Purpose_Analytics or research",
    #"Purpose_Service operation and security",
    #"Purpose_Legal requirement"

]

# convert text and labels into token IDs, attention masks, and tensors for training
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

#Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits.squeeze())
            preds.extend(probs.cpu().numpy())

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Optimize
def get_optimizer(model, base_lr, classifier_lr=1e-4, weight_decay=0.01):
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': 0.0
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': 0.0
        },
    ]
    return AdamW(optimizer_grouped_parameters)

# Train
def train_model(lr, train_loader, pos_weight_val):
    model = AutoModelForSequenceClassification.from_pretrained('nlpaueb/legal-bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([pos_weight_val]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"LR {lr:.0e} | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Final retrain
def retrain_best_model(best_lr, best_pos_weight, full_train_loader):
    model = AutoModelForSequenceClassification.from_pretrained('nlpaueb/legal-bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=best_lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([best_pos_weight]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(full_train_loader, desc=f"Retrain | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Training Loop
all_results = []

for label_to_train in custom_labels:
    safe_label = label_to_train.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_legalbert_model_{safe_label}.pt"
    val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"

    if model_path.exists():
        print(f" Skipping {label_to_train} — model already exists.")
        continue

    print(f"\nTraining for: {label_to_train}")

    train_df = sam[sam['policy_type'] == 'TRAIN'].copy()
    train_df['binary_label'] = train_df['value_name'].apply(lambda x: 1 if label_to_train in x else 0)

    positives = train_df[train_df['binary_label'] == 1]
    negatives = train_df[train_df['binary_label'] == 0]
    if len(positives) < 5:
        print(f" Too few positives ({len(positives)}). Skipping.")
        continue

    # pos_weight
    label_counts = Counter(train_df['binary_label'].tolist())
    pos_weight = label_counts[0] / label_counts[1]
    print(f" Using pos_weight: {pos_weight:.2f}")

    # Train/val
    texts = train_df['segment_text'].tolist()
    labels = train_df['binary_label'].tolist()
    texts_train, texts_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    with open(val_save_path, "wb") as f:
        pickle.dump({"texts": texts_val, "labels": y_val}, f)

    # Oversample positives
    train_data = pd.DataFrame({'segment_text': texts_train, 'binary_label': y_train})
    positives_train = train_data[train_data['binary_label'] == 1]
    negatives_train = train_data[train_data['binary_label'] == 0]

    max_oversample_size = min(len(negatives_train), len(positives_train) * 4)
    oversampled_positives = positives_train.sample(max_oversample_size, replace=True, random_state=SEED)
    train_df_balanced = pd.concat([negatives_train, oversampled_positives], ignore_index=True).sample(frac=1, random_state=SEED)

    # loaders
    train_dataset = TextDataset(train_df_balanced['segment_text'].tolist(), train_df_balanced['binary_label'].tolist())
    val_dataset = TextDataset(texts_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # find best model
    best_f1 = 0
    best_lr = None
    for lr in LEARNING_RATES:
        model = train_model(lr, train_loader, pos_weight)
        acc, f1, precision, recall = evaluate_model(model, val_loader)

        if f1 > best_f1:
            best_f1 = f1
            best_lr = lr
            torch.save(model.state_dict(), model_path)
            print(f"Best model updated and saved at LR {lr} for label {label_to_train}")

        all_results.append({
            'label': label_to_train,
            'learning_rate': lr,
            'pos_weight': round(pos_weight, 2),
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'best_model': 'best' if f1 == best_f1 else ''
        })

        # Save results after every LR
        all_models_df = pd.DataFrame(all_results)

        # Append or create "All Models" sheet
        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "All Models" in book.sheetnames:
                existing_df = pd.read_excel(excel_path, sheet_name="All Models")
                combined_df = pd.concat([existing_df, all_models_df], ignore_index=True)
                combined_df.drop_duplicates(subset=['label', 'learning_rate'], inplace=True)
            else:
                combined_df = all_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_df.to_excel(writer, sheet_name="All Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                all_models_df.to_excel(writer, sheet_name="All Models", index=False)

        # Append or create "Best Models" sheet
        best_models_df = all_models_df.sort_values('f1', ascending=False).drop_duplicates('label')

        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "Best Models" in book.sheetnames:
                existing_best_df = pd.read_excel(excel_path, sheet_name="Best Models")
                combined_best_df = pd.concat([existing_best_df, best_models_df], ignore_index=True)
                combined_best_df.drop_duplicates(subset=['label'], inplace=True)
            else:
                combined_best_df = best_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_best_df.to_excel(writer, sheet_name="Best Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                best_models_df.to_excel(writer, sheet_name="Best Models", index=False)



    # Final retrain
    full_dataset = TextDataset(train_df['segment_text'].tolist(), train_df['binary_label'].tolist())
    full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n Retraining best model for {label_to_train} on full data...")
    best_model = retrain_best_model(best_lr, pos_weight, full_loader)

    torch.save(best_model.state_dict(), model_path)
    print(f" Final model saved to {model_path}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

LR 5e-05 | Epoch 2/3:   0%|          | 0/44 [00:00<?, ?it/s]

LR 5e-05 | Epoch 3/3:   0%|          | 0/44 [00:00<?, ?it/s]

Best model updated and saved at LR 5e-05 for label Information Type_IP address and device IDs

 Retraining best model for Information Type_IP address and device IDs on full data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Retrain | Epoch 1/3:   0%|          | 0/42 [00:00<?, ?it/s]

## Test Legalbert

In [ ]:
# Config
MAX_LEN = 512
BATCH_SIZE = 32
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/models")
TEST_EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Project/models/test_results_legalbert.xlsx"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('nlpaueb/legal-bert-base-uncased')


custom_labels = [
    'Purpose_Essential service or feature',
    "Collection Process_Shared by first party with a third party",
    "Information Type_IP address and device IDs"

]


# convert text and labels into token IDs, attention masks, and tensors for training
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).squeeze(-1)
            probs = probs.detach().cpu().numpy().flatten()
            preds.extend(probs)

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Load models, test and save results
results = []

for label_to_test in tqdm(custom_labels, desc="Testing all labels"):
    safe_label = label_to_test.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_legalbert_model_{safe_label}.pt"

    if not model_path.exists():
        print(f" Model file missing for {label_to_test}, skipping.")
        continue

    print(f"\n Evaluating model for label: {label_to_test}")

    # Prepare test data
    test_df = sam[sam['policy_type'] == 'TEST'].copy()
    test_df['binary_label'] = test_df['value_name'].apply(lambda x: 1 if label_to_test in x else 0)

    test_dataset = TextDataset(test_df['segment_text'].tolist(), test_df['binary_label'].tolist())
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained('nlpaueb/legal-bert-base-uncased', num_labels=1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    acc, f1, precision, recall = evaluate_model(model, test_loader)
    print(f" Results: Acc={acc:.4f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}\n")

    results.append({
        'label': label_to_test,
        'accuracy': round(acc, 4),
        'f1_score': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
    })

# Save
df_results_legal = pd.DataFrame(results)
df_results_legal.to_excel(TEST_EXCEL_PATH, index=False)
print(f"\n Test results saved to: {TEST_EXCEL_PATH}")




df_results_legal